In [1]:
import cv2
from ultralytics import YOLO
import sys

# 1. Cargar modelo
model = YOLO("yolo11n.pt")

# 2. Abrir el video de entrada
video_path = "video_trafico.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: No se pudo abrir el video.")
    sys.exit()

# Configurar para guardar el video de salida
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
writer = cv2.VideoWriter('resultado/trafficCam_resultado.mp4',
                         cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

# --- VARIABLES PARA EL CONTEO ---
ids_contados = set() # Usamos un set para guardar IDs únicos y que no se repitan

print("Procesando video con seguimiento de IDs...")

# 3. Bucle frame a frame
frame_count = 0
while cap.isOpened():
    success, frame = cap.read()

    if success:
        # --- APLICAR TRACKING (Seguimiento) ---
        # .track mantiene el mismo ID para el mismo coche entre frames
        # persist=True es necesario para que el tracker no se reinicie
        results = model.track(frame, conf=0.5, classes=[2, 3, 5, 7], persist=True, verbose=False)

        # Comprobar si hay detecciones con ID
        if results[0].boxes.id is not None:
            # Extraer los IDs detectados en este frame
            ids = results[0].boxes.id.int().cpu().tolist()
           
            # Añadirlos al conjunto (un set solo guarda valores únicos)
            for id_vehiculo in ids:
                ids_contados.add(id_vehiculo)

        # --- VISUALIZAR ---
        frame_anotado = results[0].plot()
       
        # Dibujar el contador actual en el video (esquina superior izquierda)
        cv2.putText(frame_anotado, f"Vehiculos totales: {len(ids_contados)}",
                    (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # --- GUARDAR EN ARCHIVO ---
        writer.write(frame_anotado)
       
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"Frames procesados: {frame_count} | Vehículos detectados: {len(ids_contados)}", end="\r")
    else:
        break

# 4. Limpiar memoria
cap.release()
writer.release()

print(f"\n\nProceso finalizado con éxito.")
print(f"-----------------------------------")
print(f"TOTAL DE VEHÍCULOS ÚNICOS: {len(ids_contados)}")
print(f"-----------------------------------")
print(f"El video se ha guardado en: resultado/trafficCam_resultado.mp4")

Procesando video con seguimiento de IDs...


KeyboardInterrupt: 

In [ ]:
import cv2 
from ultralytics import YOLO 
from IPython.display import display, Image as IPImage, clear_output
from PIL import Image
import numpy as np

# 1. Cargar modelo 
model = YOLO("yolo11n.pt") 

# 2. Abrir el video de entrada 
video_path = "video_trafico.mp4" 
cap = cv2.VideoCapture(video_path) 

# Configurar para guardar el video de salida 
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) 
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) 
fps = int(cap.get(cv2.CAP_PROP_FPS)) 
writer = cv2.VideoWriter('trafficCam.mp4', 
                         cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto)) 

# Variables para conteo
total_vehiculos_detectados = 0
conteo_por_clase = {2: 0, 3: 0, 5: 0, 7: 0}  # Car, Motorcycle, Bus, Truck
nombres_clases = {2: "Coche", 3: "Moto", 5: "Autobús", 7: "Camión"}

# 3. Bucle frame a frame 
frame_count = 0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Iniciando procesamiento del video...")
print(f"Total de frames a procesar: {total_frames}")
print("-" * 50)

while cap.isOpened(): 
    success, frame = cap.read() 
    if success: 
        # --- APLICAR YOLO --- 
        # conf=0.5: Solo objetos con más del 50% de seguridad 
        # classes=[2,3,5,7]: Solo vehículos 
        results = model(frame, conf=0.5, classes=[2, 3, 5, 7]) 
        
        # --- VISUALIZAR --- 
        # .plot() dibuja las cajas en el frame automáticamente 
        frame_anotado = results[0].plot() 
        
        # --- CONTEO DE VEHÍCULOS ---
        # Obtener las detecciones del frame actual
        detecciones = results[0].boxes
        num_vehiculos_frame = len(detecciones)
        
        # Contar por tipo de vehículo
        for det in detecciones:
            clase = int(det.cls[0])
            if clase in conteo_por_clase:
                conteo_por_clase[clase] += 1
        
        total_vehiculos_detectados += num_vehiculos_frame
        
        # Guardar en el video de salida 
        writer.write(frame_anotado) 
        
        # Mostrar en pantalla (para Jupyter/Colab)
        frame_count += 1
        if frame_count % 1 == 0:  # Mostrar cada 30 frames
            clear_output(wait=True)
            frame_rgb = cv2.cvtColor(frame_anotado, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            display(pil_img)
            
            progreso = (frame_count / total_frames) * 100
            print(f"Frame: {frame_count}/{total_frames} ({progreso:.1f}%)")
            print(f"Vehículos en este frame: {num_vehiculos_frame}")
            print(f"Total acumulado: {total_vehiculos_detectados}")
        
    else: 
        break  # Fin del video 

# 4. Limpiar memoria 
cap.release() 
writer.release() 

# 5. Mostrar resultados finales
print("\n" + "=" * 50)
print("✅ PROCESAMIENTO COMPLETADO")
print("=" * 50)
print(f"\n📊 ESTADÍSTICAS FINALES:")
print(f"  • Total de frames procesados: {frame_count}")
print(f"  • Total de vehículos detectados: {total_vehiculos_detectados}")
print(f"\n🚗 DESGLOSE POR TIPO DE VEHÍCULO:")
for clase, cantidad in conteo_por_clase.items():
    print(f"  • {nombres_clases[clase]}: {cantidad}")
print("=" * 50)

AttributeError: module 'torch' has no attribute '_utils'